In [5]:
import dotenv
from agents import Agent, Runner,function_tool
import requests
import os
import openai

dotenv.load_dotenv()

True

In [6]:

def get_popular_movies():
    """인기영화 정보를 가져오는 api 호출 함수입니다.
    모든 영화 정보 리스트를 가져오기 위해 사용합니다.
    """
    return requests.get("https://nomad-movies.nomadcoders.workers.dev/movies").json()


def get_movie_details(id:int):
    """id 에 해당하는 영화의 디테일한 정보를 조회하기 위해 사용합니다.
    Args:
        id: 영화 아이디
    """
    return requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}").json()


def get_movie_credits(id:int):
    """id 에 해당하는 영화의 출연진 및 제작진 정보를 가져옵니다.
    Args:
        id: 영화 아이디
    """
    return requests.get(f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/credits").json()
    
FUNCTION_MAP = {
    "get_popular_movies":get_popular_movies,
    "get_movie_details":get_movie_details,
    "get_movie_credits":get_movie_credits
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "인기영화 정보를 가져오는 api 호출 함수입니다. 모든 영화 정보 리스트를 가져오기 위해 사용합니다.",
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "id 에 해당하는 영화의 디테일한 정보를 조회하기 위해 사용합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "영화 아이디",
                    },
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "id 에 해당하는 영화의 출연진 및 제작진 정보를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "영화 아이디",
                    },
                },
                "required": ["id"],
            },
        },
    },
]


In [14]:
import json

messages = []
client = openai.OpenAI()


def call_api(messages):
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
        )
        assistant_message = response.choices[0].message


        if assistant_message.tool_calls:
            messages.append(
                {
                    "role": "assistant",
                    "content": assistant_message.content or "",
                    "tool_calls": [
                        {
                            "id": tc.id,
                            "type": "function",
                            "function": {
                                "name": tc.function.name,
                                "arguments": tc.function.arguments,
                            },
                        }
                        for tc in assistant_message.tool_calls
                    ],
                }
            )
            for tc in assistant_message.tool_calls:
                fn = FUNCTION_MAP[tc.function.name]
                args = json.loads(tc.function.arguments)
                if "id" in args:
                    args["id"] = int(args["id"])
                result = fn(**args)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": json.dumps(result, ensure_ascii=False),
                    }
                )
            continue

        # 텍스트 응답이 있으면 출력하고 종료
        content = assistant_message.content or ""
        messages.append({"role": "assistant", "content": content})
        print(f"AI: {content}")
        break


while True:
    message = input("영화 에이전트에게 질문하세요")
    if message == "quit" or message == "q":
        break
    messages.append({"role": "user", "content": message})
    print(f"User: {message}")
    call_api(messages)

User: 나는 SF 영화를 좋아해
AI: SF 영화는 정말 흥미로운 장르입니다! 다양한 주제와 미래의 기술, 외계 생명체, 시간 여행 등 다양한 요소를 다루고 있죠. 어떤 SF 영화가 특히 마음에 드시나요? 또는 추천을 원하실까요?
User: 인셉션이랑 인터스텔라는 이미 봤어
AI: 여기 최근 인기 있는 SF 영화를 몇 편 추천해드릴게요!

1. **Mercy (2026)**  
   ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)  
   **개요:** 가까운 미래, 한 탐정이 아내 살해 혐의로 재판을 받고 있다. 그는 고급 AI 판사에게 자신의 결백을 입증하기 위해 90분의 시간을 가진다.  
   **평점:** ⭐ 7.08

2. **28 Years Later: The Bone Temple (2026)**  
   ![28 Years Later: The Bone Temple](https://image.tmdb.org/t/p/w780/kK1BGkG3KAvWB0WMV1DfOx9yTMZ.jpg)  
   **개요:** Dr. Kelson은 세상을 바꿀 수 있는 충격적인 새 관계에 휘말리게 되고, Spike는 Jimmy Crystal과의 만남에서 벗어날 수 없는 악몽에 사로잡힌다.  
   **평점:** ⭐ 7.206

3. **Greenland 2: Migration (2026)**  
   ![Greenland 2: Migration](https://image.tmdb.org/t/p/w780/z2tqCJLsw6uEJ8nJV8BsQXGa3dr.jpg)  
   **개요:** 인류를 위협하는 혜성의 공격 이후 안전한 그린란드 벙커에서 지내던 Garrity 가족이 유럽의 황무지를 횡단하는 위험한 여정을 떠난다.  
   **평점:** ⭐ 6.526

4. **Space/Time (2025)**  
   ![Space/Time](https://image.tmdb.org/t/p/w7

KeyboardInterrupt: Interrupted by user